In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import math
import pickle

from chronos import Chronos2Pipeline


# Forcing Fitting Extension

In [2]:
with open("/glade/work/stevenxu/FAIR_models/origional_model_extension.pkl", "rb") as f_in:
    f1 = pickle.load(f_in)

#with open("/glade/work/stevenxu/FAIR_models/top10_model_extension.pkl", "rb") as f_in:
 #   f2 = pickle.load(f_in)

#with open("/glade/work/stevenxu/FAIR_models/top20_model_extension.pkl", "rb") as f_in:
 #   f3 = pickle.load(f_in)

## Chronos Fitting

In [3]:
from chronos import Chronos2Pipeline
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cpu")


In [4]:
no_forcing_species = []
for specie in f1.species:
    da = f1.forcing.sel(scenario = f1.scenarios[3], specie=specie).mean(dim="config")
    if da.sum() == 0:
        no_forcing_species.append(specie)

no_forcing_species

['CO2 FFI',
 'CO2 AFOLU',
 'Sulfur',
 'BC',
 'OC',
 'NH3',
 'NOx',
 'VOC',
 'CO',
 'Equivalent effective stratospheric chlorine']

In [7]:
da = f1.forcing
da

<xarray.DataArray (timebounds: 551, scenario: 7, config: 841, specie: 61)> Size: 2GB
array([[[[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         ...,
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],

        [[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
...
         [            nan,             nan,  5.64837126e-01, ...,
          -7.18074183e-03, -2.13106204e-01,             nan],
         [            nan,             nan,  4.73988244e-01, ...,
           6.37880189e-03, -2.65986519e-01,             nan],
         [            nan,             nan,  5.03975243e-01, ...,
           9.43978386e-03, -2.64059859e-01,             nan]],

        [[            nan,             nan, -6.10161490e-01, ...,
           9.66231785e-03, -2.36981492e-01,             nan],
         [            nan,             nan, -3.55287060e-01, ...,
           1.20082983e-02, -6.03644578e-02,             nan],
         [            nan,             nan, -6.26112170e-01, ...,
           5.99079037e-03, -8.80490876e-02,             nan],
         ...,
         [            nan,             nan, -5.24402875e-01, ...,
          -7.30355152e-03, -2.13106204e-01,             nan],
         [            nan,             nan, -5.70153298e-01, ...,
           6.50488271e-03, -2.65986519e-01,             nan],
         [            nan,             nan, -4.87225304e-01, ...,
           9.69837968e-03, -2.64059859e-01,             nan]]]])
Coordinates:
  * timebounds  (timebounds) float64 4kB 1.75e+03 1.751e+03 ... 2.3e+03
  * scenario    (scenario) <U17 476B 'high-extension' ... 'verylow-overshoot'
  * config      (config) int64 7kB 1234 2451 5859 ... 1592589 1594247 1597937
  * specie      (specie) <U43 10kB 'CO2 FFI' ... 'Equivalent effective strato...

In [8]:
# Start from your xarray
full_da = f1.forcing.sel(
    scenario=f1.scenarios[3]
).mean(dim='config')

# To DataFrame: columns -> timepoints, scenario, specie, value
full_df = full_da.to_dataframe('value').reset_index()
full_df = full_df[~full_df['specie'].isin(no_forcing_species)].copy()


# Convert FAIR timepoints (1750.5, 1751.5, …) to integer years
full_df['year'] = full_df['timebounds'].astype(int)

# We only need up to 2023 for this forecast -> avoid OutOfBoundsDatetime
full_df = full_df[full_df['year'] <= 2023].copy()

# Create a proper datetime column for Chronos
# (all years here are between 1750 and 2023, so this is safe)
full_df['timebounds'] = pd.to_datetime(
    full_df['year'].astype(str),
    format="%Y"
)

# Split into train / test / future
train_df = full_df[full_df['year'] < 2000].copy()
test_df  = full_df[(full_df['year'] >= 2000) & (full_df['year'] <= 2023)].copy()

# Same timestamps, but drop the target for future_df
future_df = test_df.drop(columns='value')

prediction_length = future_df['timebounds'].nunique()

# Call Chronos on the datetime column


In [9]:
full_df

,timebounds,specie,scenario,value,year
2,1750-01-01,CO2,medium-extension,0.000000,1750
3,1750-01-01,CH4,medium-extension,0.000000,1750
4,1750-01-01,N2O,medium-extension,0.000000,1750
12,1750-01-01,CFC-11,medium-extension,0.000000,1750
13,1750-01-01,CFC-12,medium-extension,0.000000,1750
...,...,...,...,...,...
16708,2023-01-01,Aerosol-cloud interactions,medium-extension,-0.758722,2023
16709,2023-01-01,Ozone,medium-extension,0.448363,2023
16710,2023-01-01,Light absorbing particles on snow and ice,medium-extension,0.067031,2023
16711,2023-01-01,Stratospheric water vapour,medium-extension,0.055514,2023


In [11]:
pred_df = pipeline.predict_df(
    train_df,
    future_df=future_df,
    prediction_length=prediction_length,
    quantile_levels=[0.1, 0.5, 0.9],    
    id_column='specie',
    timestamp_column='timebounds',   # <-- use this, not 'year'
    target='value',
)

/glade/work/stevenxu/conda-envs/amoc-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [14]:
def plot_forecast(
    context_df: pd.DataFrame,
    pred_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target_column: str,
    timeseries_id: str,
    id_column: str = "specie",
    timestamp_column: str = "timebounds",   # <-- default now
    history_length: int = 100,
    title_suffix: str = "",
    savePlot: bool = False,
    showPlot: bool = True,
    save_path: str = '/glade/u/home/stevenxu/FAIRproject/graph_outputs/Chronos_test_plots/forcing',
):
    ts_context = (
        context_df
        .query(f"{id_column} == @timeseries_id")
        .set_index(timestamp_column)[target_column]
    )

    ts_pred = (
        pred_df
        .query(f"{id_column} == @timeseries_id and target_name == @target_column")
        .set_index(timestamp_column)[["0.1", "predictions", "0.9"]]
    )

    ts_ground_truth = (
        test_df
        .query(f"{id_column} == @timeseries_id")
        .set_index(timestamp_column)[target_column]
    )

    last_date = ts_context.index.max()
    start_idx = max(0, len(ts_context) - history_length)
    plot_cutoff = ts_context.index[start_idx]

    ts_context = ts_context[ts_context.index >= plot_cutoff]
    ts_pred = ts_pred[ts_pred.index >= plot_cutoff]
    ts_ground_truth = ts_ground_truth[ts_ground_truth.index >= plot_cutoff]

    fig = plt.figure(figsize=(8, 5))
    ax = fig.gca()

    ts_context.plot(ax=ax, label=f"historical {target_column}", color="xkcd:azure")
    ts_ground_truth.plot(ax=ax, label=f"future {target_column} (ground truth)", color="xkcd:grass green")
    ts_pred["predictions"].plot(ax=ax, label="forecast", color="xkcd:violet")

    ax.fill_between(
        ts_pred.index,
        ts_pred["0.1"],
        ts_pred["0.9"],
        alpha=0.7,
        label="prediction interval",
        color="xkcd:light lavender",
    )

    ax.axvline(x=last_date, color="black", linestyle="--", alpha=0.5)
    ax.legend(loc="upper left")
    ax.set_title(f"{target_column} forcing forecast for {timeseries_id} {title_suffix}")
    
    if showPlot:
        plt.show()

    if savePlot:
        save_dir = save_path + f'/{timeseries_id}_Chronos_Forcing_Test.png'
        fig.savefig(save_dir)
    
    plt.close(fig)
    

In [15]:
target_column = "value"

for specie in full_df['specie'].unique():
    plot_forecast(
        full_df,
        pred_df,
        test_df,
        target_column=target_column,
        timeseries_id=specie,
        title_suffix="(with covariates)",
        savePlot=True,
        showPlot=False
    )